# Spoke silver layer - rides

## Imports

In [0]:
from pyspark.sql import functions as F, Window as W

## Assigning variables

In [0]:
SOURCE = "spoke.bronze.rides"
TARGET = "spoke.silver.rides"
QUARANTINE = "spoke.silver.rides_quarantine"

## Reading bronze source

In [0]:
df_bronze = spark.table(SOURCE)

## Counting duplicate ride_id and recalculate duration for quality purposes

In [0]:
subset = [
    "bike_id",
    "device",
    "duration_seconds",
    "end_station_id",
    "ended_at",
    "issues",
    "ride_id",
    "start_station_id",
    "started_at",
    "user_type",
    "identical_records",
    "is_unclosed",
    "calculated_duration",
    "has_wrong_duration",
    "_ingested_at",
    "_source_file",
    "_ingest_batch_id",
    "_source_file_modification_time"]

identical_records = F.count("ride_id").over(W.partitionBy("ride_id"))
is_unclosed = (F.col("end_station_id").isNull() | F.col("ended_at").isNull())
calculated_duration_seconds = F.timestamp_diff("second", F.col("started_at"), F.col("ended_at"))
has_wrong_duration = F.abs(calculated_duration_seconds - F.col("duration_seconds")) > 60

df_bronze_with_metadata = (df_bronze
                           .withColumn("identical_records", identical_records)
                           .withColumn("is_unclosed", is_unclosed)
                           .withColumn("calculated_duration", calculated_duration_seconds)
                           .withColumn("has_wrong_duration", has_wrong_duration)
                           .select(subset))

## Put corrupt records in quarantine

In [0]:
df_quarantine = (df_bronze_with_metadata
                 .filter(F.col("_corrupt_record").isNotNull())
                 .withColumn("_quarantine_reasons", F.array(F.lit("corrupt_record")))
                 .withColumn("_quarantined_at", F.current_timestamp()))

In [0]:
df_to_dedup = (df_bronze_with_metadata
               .filter(F.col("_corrupt_record").isNull()))

## Deduplicating

In [0]:
subset = ["bike_id",
          "device",
          "duration_seconds",
          "end_station_id",
          "ended_at",
          "issues",
          "ride_id",
          "start_station_id",
          "started_at",
          "user_type"]

Window = (W.partitionBy(*subset)
          .orderBy(
              F.col("_source_file_modification_time").desc(),
              F.col("_source_file").desc()))

df_dedup = (df_to_dedup
             .withColumn("rank", F.row_number().over(Window))
             .filter(F.col("rank") == 1)
             .drop("rank"))

In [0]:
df_count = (df_dedup
            .groupBy("ride_id")
            .count()
            .filter(F.col("count") > 1))

In [0]:
df_quarantine = (df_quarantine
                 .unionByName(
                     df_dedup
                     .join(df_count, "ride_id", "left_semi")
                     .withColumn("_quarantine_reasons", F.array(F.lit("duplicate_ride_with_different_data")))
                     .withColumn("_quarantined_at", F.current_timestamp())))

In [0]:
df_deduped = (df_dedup
              .join(df_count, "ride_id", "left_anti")
              .select(df_dedup.columns))

## Casting

In [0]:
df_cast = (df_deduped
           .withColumn("device_app_version", F.col("device.app_version"))
           .withColumn("device_platform", F.col("device.platform"))
           .withColumn("duration_seconds_cast", F.col("duration_seconds").try_cast("int"))
           .withColumn("ended_at_cast", F.try_to_timestamp(F.col("ended_at")))
           .withColumn("started_at_cast", F.try_to_timestamp(F.col("started_at"))))

## Data quality check + quarantine reasons

In [0]:
malformed_duration_seconds = (F.col("duration_seconds_cast").isNull()
                              & F.col("duration_seconds").isNotNull())

malformed_ended_at = (F.col("ended_at_cast").isNull()
                      & F.col("ended_at").isNotNull())

malformed_started_at = (F.col("started_at_cast").isNull()
                        & F.col("started_at").isNotNull())

quarantine_reasons = F.array_compact(
    F.array(
        F.when(malformed_duration_seconds, F.lit("malformed_duration_seconds")),
        F.when(malformed_ended_at, F.lit("malformed_ended_at")),
        F.when(malformed_started_at, F.lit("malformed_started_at"))))

In [0]:
df_quarantine = (df_quarantine
                 .unionByName(df_cast
                              .withColumn("_quarantine_reasons", quarantine_reasons)
                              .filter(F.size("_quarantine_reasons") > 0)
                              .withColumn("_quarantined_at", F.current_timestamp())
                              .select(df_quarantine.columns)))

In [0]:
subset = [
    "bike_id",
    "device_app_version",
    "device_platform",
    "duration_seconds",
    "end_station_id",
    "ended_at",
    "issues",
    "ride_id",
    "start_station_id",
    "started_at",
    "user_type",
    "identical_records",
    "is_unclosed",
    "calculated_duration",
    "has_wrong_duration",
    "_ingested_at",
    "_source_file",
    "_ingest_batch_id",
    "_source_file_modification_time"]

# Retrieving similar records from bronze for each ride_id to communicate to gold
similar_records = F.count_distinct(df_bronze.ride_id).over(W.partitionBy("ride_id"))

df_silver = (df_cast
             .withColumn("_quarantine_reasons", quarantine_reasons)
             .filter(F.size("_quarantine_reasons") == 0)
             .drop("_quarantine_reasons", "duration_seconds", "ended_at", "started_at", "device")
             .withColumnRenamed("duration_seconds_cast", "duration_seconds")
             .withColumnRenamed("ended_at_cast", "ended_at")
             .withColumnRenamed("started_at_cast", "started_at")
             .select(subset))

## Load to silver

In [0]:
(df_quarantine
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(QUARANTINE))

In [0]:
(df_silver
 .write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(TARGET))